# 🔑 Solutions: Lesson 05 · PyTorch Fundamentals

> **Instructor's note.** Don't read these until you've really tried! For each problem: read the statement, look at the solution, then *close the notebook and re-implement it from memory* in the lesson notebook. The self-checks below are identical to the ones you saw there.

Each solution cell is self-contained and runs top-to-bottom.

In [1]:
import sys; sys.path.append("..")
import numpy as np
import matplotlib.pyplot as plt
from utils import dlcourse as dl
import pandas as pd, math
try:
    import torch, torch.nn as nn, torch.nn.functional as F
    from torch.utils.data import TensorDataset, DataLoader
    cpu = torch.device("cpu")
except ImportError:
    pass
dl.set_seed(42)
%matplotlib inline

---
### 🧩 Problem 5.1: Tensor gymnastics  &nbsp; `⭐`

Write `prep(images, labels)` for a batch of MNIST-style images: `images` is a float tensor `(N, 28, 28)` with values in [0, 1].
Return `(x, counts)` where `x` is the images **flattened to `(N, 784)` and standardised** with mean 0.1307 and std 0.3081,
and `counts` is a tensor of length 10 with the number of examples per class (`torch.bincount` with `minlength=10`).


In [2]:
def prep(images, labels):
    x = (images.flatten(1) - 0.1307) / 0.3081
    return x, torch.bincount(labels, minlength=10)

In [3]:
# ▶ self-check
imgs = torch.rand(20, 28, 28); labs = torch.tensor([0, 1, 1, 3, 3, 3] + [9] * 14)
out = prep(imgs, labs)
ok = out is not None
dl.check("x has shape (20, 784)", ok and out[0].shape == (20, 784))
dl.check("x is standardised correctly", ok and torch.allclose(out[0][0, 0], (imgs[0, 0, 0] - 0.1307) / 0.3081))
dl.check("class counts correct", ok and out[1].tolist() == [1, 2, 0, 3, 0, 0, 0, 0, 0, 14])

✅ x has shape (20, 784)
✅ x is standardised correctly
✅ class counts correct


---
### 🧩 Problem 5.2: Autograd on your own function  &nbsp; `⭐⭐`

Write `grad_of(f, x0)` that takes a function `f` (scalar-valued, built from torch operations) and a Python list `x0`,
and returns the **gradient of `f` at `x0` as a list of floats**, computed with autograd.
Then verify it by hand for $f(x)=\sum_i x_i^2\sin(x_i)$, whose derivative is $2x\sin x + x^2\cos x$.


In [4]:
def grad_of(f, x0):
    x = torch.tensor(x0, requires_grad=True)
    f(x).backward()
    return x.grad.tolist()

In [5]:
# ▶ self-check
f = lambda x: (x ** 2 * torch.sin(x)).sum()
pts = [0.5, 1.0, 2.0]
g = grad_of(f, pts)
truth = [2 * p * np.sin(p) + p ** 2 * np.cos(p) for p in pts]
dl.check("autograd gradient matches calculus", g is not None and np.allclose(g, truth, atol=1e-5))

✅ autograd gradient matches calculus


**Why this works.** `backward()` fills `x.grad` with ∂f/∂x for every element at once: no formulas needed.

---
### 🧩 Problem 5.3: Build an MLP factory  &nbsp; `⭐⭐`

Write `make_mlp(sizes)` returning an `nn.Sequential` with `nn.Linear` layers for consecutive sizes and a `nn.ReLU()` **between** layers
(not after the last one). e.g. `make_mlp([784, 128, 10])` → Linear(784,128), ReLU, Linear(128,10).


In [6]:
def make_mlp(sizes):
    layers = []
    for i, (a, b) in enumerate(zip(sizes[:-1], sizes[1:])):
        layers.append(nn.Linear(a, b))
        if i < len(sizes) - 2:
            layers.append(nn.ReLU())
    return nn.Sequential(*layers)

In [7]:
# ▶ self-check
m = make_mlp([784, 128, 64, 10])
ok = m is not None
dl.check("109,386 parameters (as counted in Lesson 02)", ok and dl.count_params(m) == 109386)
dl.check("output shape (4, 10)", ok and m(torch.randn(4, 784)).shape == (4, 10))
dl.check("no activation after the final layer", ok and isinstance(list(m)[-1], nn.Linear))
dl.check("exactly 2 ReLUs", ok and sum(isinstance(l, nn.ReLU) for l in m) == 2)

✅ 109,386 parameters (as counted in Lesson 02)
✅ output shape (4, 10)
✅ no activation after the final layer
✅ exactly 2 ReLUs


---
### 🧩 Problem 5.4: Train a line  &nbsp; `⭐⭐`

Write `fit_line(x, y, epochs=300, lr=0.1)` that trains `nn.Linear(1, 1)` with SGD and MSE using the five-step loop (full batch),
and returns the learned `(w, b)` as Python floats. Data: $y=3x+2+\text{noise}$; you should recover roughly $w≈3,\ b≈2$.


In [8]:
def fit_line(x, y, epochs=300, lr=0.1):
    model = nn.Linear(1, 1)
    opt = torch.optim.SGD(model.parameters(), lr=lr)
    loss_fn = nn.MSELoss()
    for _ in range(epochs):
        loss = loss_fn(model(x), y)
        opt.zero_grad(); loss.backward(); opt.step()
    return model.weight.item(), model.bias.item()

In [9]:
# ▶ self-check
g = torch.Generator().manual_seed(0)
x = torch.rand(100, 1, generator=g) * 4 - 2
y = 3 * x + 2 + 0.1 * torch.randn(100, 1, generator=g)
res = fit_line(x, y)
dl.check("learned w ≈ 3 and b ≈ 2", res is not None and abs(res[0] - 3) < 0.15 and abs(res[1] - 2) < 0.15, "did you call zero_grad, backward and step?")

✅ learned w ≈ 3 and b ≈ 2


---
### 🧩 Problem 5.5: Hunt the two bugs  &nbsp; `⭐⭐`

This `train_step` is supposed to perform one correct update, but it contains **two classic bugs** (both discussed above).
Find and fix them. The self-check compares two consecutive steps with a reference implementation.


In [10]:
def train_step(model, loss_fn, optimizer, xb, yb):
    out = model(xb)
    loss = loss_fn(out, yb)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    return loss.item()

In [11]:
# ▶ self-check
import copy
torch.manual_seed(0)
base = nn.Linear(3, 4); xb = torch.randn(16, 3); yb = torch.randint(0, 4, (16,))
m1, m2 = copy.deepcopy(base), copy.deepcopy(base)
o1, o2 = torch.optim.SGD(m1.parameters(), lr=0.1), torch.optim.SGD(m2.parameters(), lr=0.1)
ce = nn.CrossEntropyLoss()
for _ in range(2):
    train_step(m1, ce, o1, xb, yb)                         # yours
    o2.zero_grad(); ce(m2(xb), yb).backward(); o2.step()   # reference
dl.check("two consecutive steps match the reference", torch.allclose(m1.weight, m2.weight, atol=1e-6),
         "still different? check BOTH the softmax and zero_grad")

✅ two consecutive steps match the reference


**Why this works.** With the stray softmax the loss sees probabilities instead of logits (weak, distorted gradients); without `zero_grad` the second step reuses the first step's gradient on top of its own.